# Step 3D: evaluate the LLM-assisted classifier

This notebook is read-only and never sends API requests. It evaluates saved pilot results, not the full 5,023-ad corpus. Dry-run rows are plans, not model predictions. Rule-based labels and optional AI-assisted suggestions are comparison references, never independently verified ground truth.

Start with `python -m src.run_ai_classification --provider openai --model gpt-4.1-mini --max-ads 30 --input-usd-per-million 0.40 --output-usd-per-million 1.60 --budget-usd 1 --dry-run`. Configure credentials and use the explicit `--execute` flag only after reviewing the cost forecast. See README for resume and provider configuration.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'data').is_dir())
sys.path.insert(0, str(ROOT))
from src.validate_classification import HUMAN_FIELDS, evaluate_review, read_csv, sha256
from src.classify_jobs_v2 import CATEGORIES, RELEVANCES
RUN_DIR = None  # Set explicitly to retain a particular pilot run.
if RUN_DIR is None:
    candidates = list((ROOT / 'data' / 'processed').glob('step3d-*/run_summary.json'))
    if not candidates:
        raise FileNotFoundError('Run the dry-run pilot command first')
    run_dir = max(candidates, key=lambda p: p.parent.name).parent
else:
    run_dir = Path(RUN_DIR)
summary = json.loads((run_dir / 'run_summary.json').read_text(encoding='utf-8'))
plan = json.loads((run_dir / 'plan.json').read_text(encoding='utf-8'))
if sha256(run_dir / 'pilot_input.csv') != plan['pilot_input_sha256']:
    raise ValueError('Frozen pilot input changed')
rows = read_csv(run_dir / 'pilot_results.csv')
completed = rows.loc[rows.ai_status.isin(['completed', 'cache_hit'])].copy()
print('Pilot folder:', run_dir)
print('Run mode/status:', summary['mode'], '/', summary['status'])
print('Pilot rows:', len(rows), '| completed model predictions:', len(completed))
display(rows.ai_status.value_counts().rename_axis('AI status').to_frame('Advertisements'))

## Model, prompt, tokens and cost

Expected counts use a character heuristic; the conservative pre-request reservation uses UTF-8 payload bytes plus protocol overhead and the output cap. Actual provider token counts are recorded when available. Missing usage, HTTP errors and unknown-outcome timeouts retain their full reservations. Retried requests count toward the same run budget. Cached decisions can appear without new requests or costs.

Configured prices are estimates in USD per million tokens, not an invoice guarantee. Provider fees and changed pricing may differ. A dry run has no provider-reported token usage and makes no paid requests. The pilot is balanced by the existing rule-based relevance classes (seed 47), rather than being representative of population prevalence.

In [ ]:
display(pd.Series({
    'provider': summary['provider'], 'model': summary['model'],
    'prompt_version': summary['prompt_version'], 'policy_sha256': summary['policy_sha256'],
    'budget_usd': summary['budget_usd'], 'accounted_cost_usd': summary['accounted_cost_usd'],
    'attempts_reserved_total': summary['attempts_reserved_total'],
    'reported_input_tokens': summary['reported_input_tokens'],
    'reported_output_tokens': summary['reported_output_tokens'],
    'unknown_or_unreported_cost_attempts': summary['unknown_or_unreported_cost_attempts'],
}, name='Value').to_frame())
display(pd.Series(summary['forecast'], name='Forecast').to_frame())
display(rows.relevance.value_counts().rename_axis('Rule-based pilot stratum').to_frame('Advertisements'))
print('Prices:', summary['pricing_usd_per_million'])
if summary['mode'] == 'dry_run':
    print('Dry run only: no new LLM requests. Forecast tokens are not observed model usage.')

## Rule-based versus LLM classifications

Differences and uncertain cases resolved are descriptive comparisons, not accuracy. Only validated completed model outputs or cached model outputs enter these tables. Missing-description local abstentions, failed requests, budget stops and dry-run placeholders are shown separately as operational coverage; they are not silently counted as successful LLM decisions.

In [ ]:
if completed.empty:
    print('No completed LLM classifications. Disagreements and resolved-case counts are unavailable until execution.')
    disagreements = completed.copy()
else:
    completed['relevance_disagreement'] = completed.relevance.ne(completed.ai_relevance)
    completed['category_disagreement'] = completed.primary_category.ne(completed.ai_primary_category)
    completed['rule_uncertain_resolved'] = completed.relevance.eq('uncertain') & completed.ai_relevance.isin(['relevant', 'irrelevant'])
    print('Rule/AI relevance disagreements:', completed.relevance_disagreement.sum())
    print('Rule/AI primary-category disagreements:', completed.category_disagreement.sum())
    print('Rule-uncertain cases resolved by AI:', completed.rule_uncertain_resolved.sum())
    disagreements = completed.loc[completed.relevance_disagreement | completed.category_disagreement].copy()
    display(disagreements[['advertisement_id', 'job_title', 'relevance', 'primary_category',
                           'ai_relevance', 'ai_primary_category', 'ai_reasoning', 'ai_evidence']])
    display(pd.crosstab(completed.relevance, completed.ai_relevance,
                       rownames=['Rule-based prediction'], colnames=['LLM prediction']))

## Category distribution, conflicts and evidence

Primary and secondary categories must be in the policy taxonomy; unresolved primary roles remain empty/null. Qualitative confidence is not a calibrated probability. Review-required decisions, rule/AI disagreements, duplicate clusters and hybrid roles should be inspected before any counting policy changes. Original and AI-derived reporting scopes remain separate.

In [ ]:
if completed.empty:
    print('AI category distributions and evidence are unavailable: no model output exists yet.')
else:
    display(completed.ai_relevance.value_counts().to_frame('Advertisements'))
    display(completed.ai_primary_category.replace('', '(null)').value_counts().to_frame('Advertisements'))
    if 'ai_role_scope' in completed:
        display(completed.ai_role_scope.value_counts().to_frame('Advertisements'))
    needs_review = completed.ai_review_required.str.casefold().eq('true')
    conflict = completed.relevance.ne(completed.ai_relevance) | completed.primary_category.ne(completed.ai_primary_category)
    display(completed.loc[needs_review | conflict, ['advertisement_id', 'job_title', 'ai_relevance',
        'ai_primary_category', 'ai_secondary_categories', 'ai_confidence', 'ai_review_required', 'ai_reasoning', 'ai_evidence']])

def inspect_ai_ad(advertisement_id):
    matching = rows.loc[rows.advertisement_id.eq(str(advertisement_id))]
    if len(matching) != 1:
        raise ValueError('Choose an ID from this pilot')
    row = matching.iloc[0]
    print('Title:', row.job_title, '| ID:', row.advertisement_id, '| AI status:', row.ai_status)
    print('Rules:', row.relevance, '| category:', row.primary_category or '(null)')
    print('AI:', row.ai_relevance or '(not classified)', '| category:', row.ai_primary_category or '(null)')
    print('AI decision explanation:', row.ai_reasoning)
    print('AI verbatim supporting excerpt:', row.ai_evidence)
    print('FULL ORIGINAL DESCRIPTION')
    print(row.description_text)

# inspect_ai_ad(rows.advertisement_id.iloc[0])

## Optional provisional AI-assisted review reference

If `manual_review_representative_ai_suggestions.csv` is supplied to the runner, its schema is inspected and mapped explicitly. Only ID and provisional relevance/category fields are imported. Any human_verified/reviewer fields in that AI-generated file are discarded. Suggestions are never included in the classifier prompt and are not ground truth. Add an unavailable file with `--suggestions PATH` in a new planned run; changing a reference changes resume identity.

This section displays label differences only. It never calculates accuracy, precision, recall or F1 against AI-generated reference labels.

In [ ]:
if 'provisional_ai_reference_relevance' not in rows:
    print('No AI-suggestions reference file was available. It is optional.')
else:
    referenced = completed.loc[completed.provisional_ai_reference_relevance.ne('')].copy()
    print('Completed pilot rows with provisional AI reference:', len(referenced))
    if len(referenced):
        referenced['provisional_reference_disagreement'] = referenced.ai_relevance.ne(referenced.provisional_ai_reference_relevance)
        print('Provisional-reference relevance disagreements (not errors):', referenced.provisional_reference_disagreement.sum())
        display(referenced[['advertisement_id', 'job_title', 'ai_relevance', 'ai_primary_category',
            'provisional_ai_reference_relevance', 'provisional_ai_reference_category', 'ai_evidence']])

## Future human validation: verified labels only

The runner creates a separate `human_review_pilot.csv`. Edit only human_relevance, human_primary_category, reviewer, review_notes and human_verified in that file. Read the full advertisement and classification policy. Use exact category labels; leave primary category blank for human irrelevant/uncertain relevance or an unresolved relevant category. Set human_verified=true only after independent human inspection. The runner never automatically verifies labels or overwrites this human-review copy on resume.

A different verified-human CSV may be selected below; only matching pilot IDs are used. The known AI-suggestions reference path/file is explicitly rejected as human ground truth. Source/AI labels must never be copied into verified truth without a human checking them. Predictions remain separate from labels.

In [ ]:
HUMAN_REVIEW_PATH = None  # Default: this run's independently editable human_review_pilot.csv
human_path = Path(HUMAN_REVIEW_PATH) if HUMAN_REVIEW_PATH else run_dir / 'human_review_pilot.csv'
reference_path = Path(plan['suggestions_path']).resolve() if plan.get('suggestions_path') else None
if 'ai_suggestions' in human_path.name.casefold() or (reference_path and human_path.resolve() == reference_path):
    raise ValueError('AI-generated suggestions are not independently verified human ground truth')
humans = read_csv(human_path)
if set(HUMAN_FIELDS + ['advertisement_id']) - set(humans):
    raise ValueError('Human review file lacks required fields')
if humans.advertisement_id.duplicated().any():
    raise ValueError('Duplicate human review IDs')
humans = humans.loc[humans.advertisement_id.isin(rows.advertisement_id)].copy()
verification = humans.human_verified.str.strip().str.casefold()
if set(verification) - {'', 'true', 'false'}:
    raise ValueError('Use blank, true or false for human_verified')
print('Pilot human rows available:', len(humans))
print('Explicitly human-verified rows:', verification.eq('true').sum())
print('Pilot rows not human-verified:', len(rows) - verification.eq('true').sum())
print('Allowed categories:')
print(*CATEGORIES, sep=chr(10))

## Precision, recall, F1 and confusion matrices after verified human review

Metrics are calculated only for completed model predictions joined to explicit verified human labels. The completed-model coverage is reported; unavailable/failed API outcomes are not disguised as predictions. Verified human uncertain judgments remain unresolved and are excluded. For relevance, predicted uncertain is an abstention/error. Category metrics use human-relevant cases with resolved categories; predicted irrelevant/uncertain/null counts as category abstention/error. Spurious relevance on human-irrelevant rows is measured in relevance metrics.

The pilot was balanced across rule predictions and may contain correlated duplicate ads. These are reviewed-pilot metrics, not population or independent unique-vacancy accuracy. Undefined scores are null; macro averages use supported human classes. Never pool provisional AI labels into these calculations.

In [ ]:
human_metrics = None
if completed.empty:
    print('No completed model predictions; human-validation metrics are unavailable.')
else:
    evaluation = completed[['advertisement_id', 'ai_relevance', 'ai_primary_category']].merge(
        humans[['advertisement_id'] + HUMAN_FIELDS], on='advertisement_id', how='left', validate='one_to_one').fillna('')
    evaluation['relevance'] = evaluation.ai_relevance
    evaluation['primary_category'] = evaluation.ai_primary_category
    evaluation['sampling_design'] = 'balanced_step3d_pilot_completed_model_rows_only'
    human_metrics = evaluate_review(evaluation)
    print('Completed model coverage:', len(completed), '/', len(rows))
    print(human_metrics['status'])
    display(pd.Series(human_metrics['counts'], name='Value').to_frame())
    for task in ['relevance', 'primary_category']:
        print('Human-validated task:', task)
        value = human_metrics[task]
        if value is None:
            print('Metrics unavailable: no verified resolved human truth for this task.')
        else:
            display(pd.Series(value['summary'], name='Value').to_frame())
            display(value['per_class'])
            display(value['confusion_matrix'])

## Next decision

Inspect disagreements, low-confidence decisions and exact evidence before expanding a pilot. A resolved AI prediction is not a verified improvement. The CLI intentionally refuses more than 30 advertisements and corpus-sized inputs. Running all 5,023 advertisements requires a separately approved design, budget and validation decision; this notebook never starts that work. Original Step 3/3B/3C data and rules are preserved.